# Point Taken — Argument-Coaching Study: Analysis

This notebook reproduces the analyses for the argument-coaching experiment from the study's data export. It computes the confirmatory test (whether the treatment condition improves in argument quality across rounds relative to control), the secondary and per-dimension analyses, the LLM-as-judge validation (agreement with human coding), and the corresponding figures.

The confirmatory analyses follow the preregistered plan (`docs/ANALYSIS_PLAN.md`). Sections beyond the primary and secondary tests are exploratory.

**Usage:** set `URL` and `TOKEN` (or `FILE`) in the configuration cell and run all cells.

In [ ]:
# --- config ---
URL   = "https://dp5wkeqgw25kj.cloudfront.net"   # live study origin
TOKEN = None   # set to E18_EXPORT_TOKEN, or leave None and use FILE
FILE  = None   # path to a saved e18_export.json instead of pulling live

import json, urllib.request
from collections import defaultdict
import pandas as pd
import numpy as np

def load_export():
    if FILE:
        return json.load(open(FILE))
    assert URL and TOKEN, "set TOKEN (or FILE)"
    u = URL.rstrip('/') + '/argument-coach/export?token=' + TOKEN
    with urllib.request.urlopen(u, timeout=60) as r:
        return json.load(r)

data = load_export()
{k: len(v) for k, v in data.items()}

## 1. Construct the per-round scored frame

In [ ]:
def pj(v):
    if isinstance(v, (dict, list)): return v
    try: return json.loads(v) if v else {}
    except Exception: return {}

cond = {c['user_id']: c.get('condition') for c in data.get('conditions', [])}
HELP = ['reasoning_quality','engagement','constructive_stance']
COUNTER = ['fallacies','evasion','hostility']

rows = []
for s in data.get('round_scores', []):
    rat = pj(s.get('rationale'))
    helpful = [s.get(d) for d in HELP if s.get(d) is not None]
    composite = s.get('composite')
    if composite is None and helpful:
        composite = float(np.mean(helpful))
    counter = [rat.get(d, {}).get('score') if isinstance(rat.get(d), dict) else rat.get(d) for d in COUNTER]
    counter = [c for c in counter if c is not None]
    rows.append({
        'user_id': s['user_id'],
        'condition': cond.get(s['user_id'], 'UNKNOWN'),
        'round': int(s['round_number']),
        'composite': composite,
        'overall': rat.get('analysis_overall',
                           (sum(helpful) - sum(counter)) if (helpful and counter) else np.nan),
        **{d: s.get(d) for d in HELP},
        **{d: (rat.get(d, {}).get('score') if isinstance(rat.get(d), dict) else rat.get(d)) for d in COUNTER},
        'model_name': s.get('model_name'),
    })
df = pd.DataFrame(rows).sort_values(['user_id','round']).reset_index(drop=True)
print(f"{df['user_id'].nunique()} participants, {len(df)} scored rounds")
df.head(10)

## 2. Prespecified exclusions

Participants who did not complete all rounds are excluded. Any scored round lacking a `model_name` is flagged as a possible fallback to canned text. Set `N_ROUNDS` to match the study configuration.

In [ ]:
N_ROUNDS = 3
completed = df.groupby('user_id')['round'].nunique()
keep = completed[completed >= N_ROUNDS].index
excluded = set(df['user_id']) - set(keep)
print(f"excluded {len(excluded)} (incomplete): {sorted(excluded)}")
fallback = df[df['model_name'].isna()]
if len(fallback):
    print(f"WARNING: {len(fallback)} scored rounds have no model_name (possible fallback)")
df = df[df['user_id'].isin(keep)].copy()
df['condition'].value_counts()

## 3. Descriptive statistics: mean score by round and condition

In [ ]:
cell = df.groupby(['condition','round'])['composite'].agg(['mean','std','count']).round(3)
display(cell)
df.groupby(['condition','round'])['overall'].mean().round(3).unstack()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,4))
for m, a in zip(['composite','overall'], ax):
    for c, g in df.groupby('condition'):
        s = g.groupby('round')[m].mean()
        a.plot(s.index, s.values, marker='o', label=f"{c} (n={g['user_id'].nunique()})")
    a.set_xlabel('Round'); a.set_ylabel(m); a.set_title(f'{m} by round'); a.legend()
    a.set_xticks(range(1, N_ROUNDS+1))
plt.tight_layout(); plt.savefig('trajectory.png', dpi=150); plt.show()

## 4. Primary confirmatory analysis: condition × round interaction

Linear mixed-effects model `composite ~ condition * round_c + (1 | participant)`. The **condition × round** interaction corresponds to H1; a positive coefficient indicates that the treatment condition improves more per round than control. If the mixed model does not converge at the available sample size, the analysis falls back to a Welch two-sample *t*-test on the round-1-to-last change score (a difference-in-differences contrast), as prespecified.

In [ ]:
d = df.copy()
d['round_c'] = d['round'] - 1                      # 0,1,2
d['is_coach'] = (d['condition'] == 'AICoach').astype(int)
try:
    import statsmodels.formula.api as smf
    md = smf.mixedlm('composite ~ is_coach * round_c', d, groups=d['user_id'])
    mf = md.fit(reml=False)
    print(mf.summary())
    print('\nH1 interaction (is_coach:round_c):',
          round(mf.params.get('is_coach:round_c', float('nan')), 4),
          'p=', round(mf.pvalues.get('is_coach:round_c', float('nan')), 4))
except Exception as e:
    print('mixed model unavailable/failed, using DiD t-test:', e)
    from scipy import stats
    ch = (d.sort_values('round').groupby(['user_id','condition'])['composite']
            .agg(lambda x: x.iloc[-1] - x.iloc[0]).reset_index())
    a = ch[ch.condition=='AICoach']['composite']; b = ch[ch.condition=='Control']['composite']
    print('mean change AICoach=%.3f Control=%.3f  DiD=%.3f' % (a.mean(), b.mean(), a.mean()-b.mean()))
    print('Welch t=%.3f p=%.4f' % stats.ttest_ind(a, b, equal_var=False))

## 5. Secondary analysis: counterproductive behaviors (H3)

Tests whether coaching reduces the summed counterproductive score (fallacies, evasion, hostility) across rounds relative to control.

In [ ]:
d['counter_sum'] = d[COUNTER].sum(axis=1)
display(d.groupby(['condition','round'])['counter_sum'].mean().round(3).unstack())
try:
    import statsmodels.formula.api as smf
    print(smf.mixedlm('counter_sum ~ is_coach * round_c', d, groups=d['user_id']).fit(reml=False).summary())
except Exception as e:
    print('skip:', e)

## 6. Exploratory analysis: per-dimension trajectories

Per-dimension means by round and condition, to identify which dimensions are most affected. Descriptive only; not corrected for multiple comparisons.

In [ ]:
for dim in HELP + COUNTER:
    piv = df.groupby(['condition','round'])[dim].mean().round(2).unstack()
    print(f"\n{dim}"); print(piv)

## 7. LLM-as-judge validation

Because the primary outcome depends on the LLM judge, its scores are validated against expert human coding. This cell exports a random subsample of rounds for an expert to score on the same six-dimension, 1–5 rubric. The subsequent cell loads the completed codes and computes agreement between the LLM and human scores (Krippendorff's α), to be benchmarked against human–human agreement on the overlapping subset.

In [ ]:
SAMPLE_N = 25
rounds = {(r['user_id'], int(r['round_number'])): r for r in data['rounds']}
samp = df.sample(min(SAMPLE_N, len(df)), random_state=7)[['user_id','round']]
out = []
for _, r in samp.iterrows():
    rr = rounds.get((r['user_id'], r['round']), {})
    out.append({'user_id': r['user_id'], 'round': r['round'],
                'topic': rr.get('topic',''), 'arguments': rr.get('arguments','')})
sample_df = pd.DataFrame(out)
for d_ in HELP + COUNTER:
    sample_df[d_+'_human'] = ''   # expert fills 1-5
sample_df.to_csv('rubric_validation_sample.csv', index=False)
print('wrote rubric_validation_sample.csv for expert coding'); sample_df.head()

In [ ]:
# after an expert fills rubric_validation_sample.csv, compute agreement:
def krippendorff_alpha_interval(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b)); a, b = a[ok], b[ok]
    if len(a) < 2: return np.nan
    Do = np.mean((a - b) ** 2)
    vals = np.concatenate([a, b]); n = len(vals)
    De = sum((vals[i]-vals[j])**2 for i in range(n) for j in range(n)) / (n*(n-1))
    return 1 - Do/De if De else np.nan

try:
    h = pd.read_csv('rubric_validation_sample.csv')
    merged = h.merge(df, on=['user_id','round'], suffixes=('_h',''))
    for dim in HELP + COUNTER:
        col = dim + '_human'
        if col in h and h[col].notna().any():
            print(dim, 'alpha=', round(krippendorff_alpha_interval(merged[col], merged[dim]), 3))
except FileNotFoundError:
    print('fill rubric_validation_sample.csv first')